# 2026-09-10 — celltype x direction per-stage log2FC heatmaps

每个 **celltype x direction** 一张 PDF heatmap。

- 行：该 celltype-direction 的 anchor 交集基因（来源 `2026-09-06_1_fate_module_celltype_source.csv` 的 `anchor_gene_list`）。
- 列：`Early, Mid, Late | Young, Old`（胚胎 | 成年，中间小分隔）。**某一期整列无 log2FC 时，该列不画**。
- 值：DESeq2 每期 **有符号 log2FoldChange**（`2026-09-01_1_five_stage_trajectory_refined.csv`），不做 z-score。
- 行按 **subclass x module** 分块，块内聚类并显示聚类树枝。
- topAnnotation：**Cell type** + **Stage**。leftAnnotation：**Subclass** + **Module**（左 subclass、右 module）。
- 注解只上色；图例写全名，**含 Module 图例**。配色沿用 `color.ipynb`（celltype / stage / subclass），module 用固定表 `2026-09-06_module_colors.csv`。
- 所有 heatmap 的矩阵合并导出为一张长表 CSV。
- 输出 PDF 到 `Plot/`，并内联显示在 notebook。

In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(readr)
  library(stringr)
  library(ComplexHeatmap)
  library(circlize)
  library(grid)
})

date_str <- "2026-09-10"
data_dir <- "/project/work/2026-09-01_GSEA_test/Data/"
plot_dir <- "/project/work/2026-09-01_GSEA_test/Plot/"

source_file  <- file.path(data_dir, "2026-09-06_1_fate_module_celltype_source.csv")
refined_file <- "/project/work/2026-09-01_DEG_summary/Data/2026-09-01_1_five_stage_trajectory_refined.csv"
module_col_file <- file.path(data_dir, "2026-09-06_module_colors.csv")

celltype_order <- c("Rod", "Cone", "AC", "BC", "HC", "MG")
fate_order     <- c("DM", "DAL", "AAM")
stage_order    <- c("Early", "Mid", "Late", "Young", "Old")
period_order   <- c("Embryonic", "Adult")
period_of_stage <- c(Early="Embryonic", Mid="Embryonic", Late="Embryonic", Young="Adult", Old="Adult")

## 配色：来自 color.ipynb
cols.celltype <- c('#A6CEE3','#1F78B4','#B2DF8A','#33A02C','#FB9A99','#E31A1C')
names(cols.celltype) <- celltype_order
cols.stage.full <- c(Early='#59A14F', Mid='#EDC948', Late='#B07AA1', Young='#4E79A7', Old='#E15759')
cols.subclass.short <- c(DM='#66C2A5', DAL='#FC8D62', AAM='#8DA0CB')  # 2026-10-19 更新；旧版 DM=#3C5488, DAL=#E64B35, AAM=#00A087

## module 固定配色
mod_col_tbl <- read_csv(module_col_file, show_col_types = FALSE)
cols.module <- setNames(mod_col_tbl$color, mod_col_tbl$module)

## direction -> 短标签
dir_short <- c("Macular direction" = "Central", "Peripheral direction" = "Peripheral")
cat("配色就绪。module 数:", length(cols.module), "\n")

In [ ]:
# ---- 读取 anchor 源表与 per-stage log2FC 表 ----------------------------------
anchor_long <- read_csv(source_file, show_col_types = FALSE) %>%
  mutate(subclass = fate) %>%
  select(direction, subclass, celltype, module, anchor_gene_list) %>%
  mutate(gene = str_split(anchor_gene_list, ";")) %>%
  unnest(gene) %>%
  select(direction, subclass, celltype, module, gene) %>%
  distinct()

logfc_tbl <- read_csv(refined_file, show_col_types = FALSE) %>%
  select(celltype, gene, direction_class,
         Early_log2FoldChange, Mid_log2FoldChange, Late_log2FoldChange,
         Young_log2FoldChange, Old_log2FoldChange)

cat("anchor gene 行数:", nrow(anchor_long), "\n")
cat("log2FC 表行数:", nrow(logfc_tbl), "\n")
cat("direction:", paste(unique(anchor_long$direction), collapse = ", "), "\n")

In [ ]:
# ---- 构建某 celltype x direction 的矩阵与注释 --------------------------------
stage_lfc_cols <- paste0(stage_order, "_log2FoldChange")

build_blocks <- function(one_direction, one_ct) {
  blk <- anchor_long %>%
    filter(direction == one_direction, celltype == one_ct) %>%
    distinct(subclass, module, gene)

  val <- logfc_tbl %>%
    filter(celltype == one_ct, direction_class == one_direction) %>%
    select(gene, all_of(stage_lfc_cols))

  blk %>% left_join(val, by = "gene")
}

# 某 celltype x direction 中，至少有一个基因有值的 stage（用于决定画哪些列）
present_stages <- function(m) {
  keep <- sapply(stage_lfc_cols, function(c) any(!is.na(m[[c]])))
  stage_order[keep]
}

make_heatmap <- function(one_direction, one_ct) {
  m <- build_blocks(one_direction, one_ct)
  if (nrow(m) == 0) return(NULL)

  # 只保留该组合实际存在 log2FC 的 stage 列
  stages_keep <- present_stages(m)
  cols_keep   <- paste0(stages_keep, "_log2FoldChange")

  m <- m %>%
    mutate(
      subclass = factor(subclass, levels = fate_order),
      module   = factor(module)
    ) %>%
    arrange(subclass, module)

  present_modules <- unique(as.character(m$module))
  mod_levels      <- mod_col_tbl$module[mod_col_tbl$module %in% present_modules]
  m <- m %>%
    mutate(module = factor(as.character(module), levels = mod_levels)) %>%
    arrange(subclass, module)

  block_chr <- paste(as.character(m$subclass), as.character(m$module), sep = " | ")
  m$block   <- factor(block_chr, levels = unique(block_chr))

  mat <- as.matrix(m[, cols_keep])
  colnames(mat) <- stages_keep
  rownames(mat) <- make.unique(m$gene)

  top_anno <- HeatmapAnnotation(
    `Cell type` = factor(rep(one_ct, length(stages_keep)), levels = one_ct),
    Stage       = factor(stages_keep, levels = stages_keep),
    col = list(`Cell type` = cols.celltype, Stage = cols.stage.full),
    show_annotation_name = FALSE,
    show_legend = TRUE
  )
  left_anno <- rowAnnotation(
    Subclass = m$subclass,
    Module   = m$module,
    col = list(Subclass = cols.subclass.short, Module = cols.module),
    show_annotation_name = FALSE,
    show_legend = TRUE
  )

  col_fun <- colorRamp2(c(-3, 0, 3), c("#2166AC", "white", "#B2182B"))

  Heatmap(
    mat,
    name = "log2FC",
    col  = col_fun,
    top_annotation = top_anno,
    left_annotation = left_anno,
    column_split = factor(period_of_stage[stages_keep],
                          levels = period_order[period_order %in% period_of_stage[stages_keep]]),
    column_gap = unit(3, "mm"),
    row_split = m$block,
    cluster_rows = TRUE,
    cluster_row_slices = FALSE,
    cluster_columns = FALSE,
    show_row_names = TRUE,
    row_names_side = "right",
    show_column_names = TRUE,
    column_names_rot = 0,
    column_names_centered = TRUE,
    show_row_dend = TRUE,
    row_dend_width = unit(10, "mm"),
    row_title = NULL,
    column_title = NULL,
    row_gap = unit(1.5, "mm"),
    border = TRUE,
    na_col = "grey90"
  )
}

In [ ]:
# ---- 主循环：每个 celltype x direction 一张 PDF + 合并矩阵 CSV ---------------
n_done <- 0
merged_list <- list()

for (one_direction in c("Macular direction", "Peripheral direction")) {
  for (one_ct in celltype_order) {
    m_probe <- build_blocks(one_direction, one_ct)
    if (nrow(m_probe) == 0) {
      cat("无 anchor 基因，跳过:", dir_short[[one_direction]], one_ct, "\n"); next
    }

    ht <- make_heatmap(one_direction, one_ct)
    n_rows   <- nrow(m_probe)
    n_blocks <- nrow(distinct(m_probe, subclass, module))

    # 汇总进合并表（保留全部 5 期，NA 表示该期无 log2FC）
    merged_list[[length(merged_list) + 1]] <- m_probe %>%
      mutate(
        direction = dir_short[[one_direction]],
        celltype  = one_ct
      ) %>%
      select(direction, celltype, subclass, module, gene, all_of(stage_lfc_cols))

    hh <- max(3.2, 1.2 + n_rows * 0.24)
    ww <- 6.5

    pdf_file <- file.path(plot_dir, paste0(
      date_str, "_celltype_direction_log2FC_heatmap_", one_ct, "_", dir_short[[one_direction]], ".pdf"
    ))
    pdf(pdf_file, width = ww, height = hh)
    draw(ht, merge_legend = TRUE,
         heatmap_legend_side = "right", annotation_legend_side = "right")
    dev.off()
    n_done <- n_done + 1
    cat(sprintf("已输出: %s  (genes=%d, blocks=%d, stages=%s, size=%.1fx%.1f)\n",
                basename(pdf_file), n_rows, n_blocks,
                paste(present_stages(m_probe), collapse = "/"), ww, hh))
  }
}

# ---- 合并矩阵 CSV -----------------------------------------------------------
merged_matrix <- bind_rows(merged_list) %>%
  mutate(direction = factor(direction, levels = c("Central", "Peripheral")),
         celltype  = factor(celltype,  levels = celltype_order),
         subclass  = factor(subclass,  levels = fate_order)) %>%
  arrange(direction, celltype, subclass, module, gene)

csv_file <- file.path(data_dir, paste0(date_str, "_celltype_direction_log2FC_matrix.csv"))
write_csv(merged_matrix, csv_file)
cat("\n已输出合并矩阵 CSV:", basename(csv_file), " 行数:", nrow(merged_matrix), "\n")
cat("完成，共输出", n_done, "张 PDF。\n")

## 追加：为合并矩阵 CSV 补充 per-stage padj

在已生成的 `2026-09-10_celltype_direction_log2FC_matrix.csv` 上追加 5 列
`{Early,Mid,Late,Young,Old}_padj`（来自 refined 表，按 celltype + gene + direction 对齐）。
只读取现有 CSV 与 refined 表，无需重跑整个绘图脚本。

In [ ]:
# ---- 追加 padj 列到合并矩阵 CSV（最简命令，不重跑绘图）-----------------------
suppressPackageStartupMessages({library(dplyr); library(readr)})

mat_f <- file.path(data_dir, paste0(date_str, "_celltype_direction_log2FC_matrix.csv"))
ref_f <- refined_file

key <- c(Central = "Macular direction", Peripheral = "Peripheral direction")

mat <- read_csv(mat_f, show_col_types = FALSE)
# 幂等：若已存在 padj 列则先移除，避免重复运行产生 .x/.y 后缀
mat <- mat %>% select(-matches("_padj(\\.x|\\.y)?$"))
ref <- read_csv(ref_f, show_col_types = FALSE) %>%
  select(celltype, gene, direction_class, all_of(paste0(stage_order, "_padj")))

out <- mat %>%
  mutate(direction_class = key[direction]) %>%
  left_join(ref, by = c("celltype", "gene", "direction_class")) %>%
  select(-direction_class)

write_csv(out, mat_f)
cat("行数:", nrow(out), " 列数:", ncol(out), "\n")
cat("列名:", paste(names(out), collapse = ", "), "\n")